# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
#import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re
import difflib
import warnings

#ignore warnings
warnings.filterwarnings("ignore")

## Data Loading and Inspection

In [2]:
aviation_df = pd.read_csv("./data/AviationData.csv", encoding="latin1", low_memory=False)
states_df = pd.read_csv("./data/USState_Codes.csv")

In [3]:
aviation_df.head(5)

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [4]:
states_df.head(5)

,US_State,Abbreviation
0,Alabama,AL
1,Alaska,AK
2,Arizona,AZ
3,Arkansas,AR
4,California,CA


### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [5]:
print("Shape:", aviation_df.shape)
print("Duplicate Event.Ids:", aviation_df["Event.Id"].duplicated().sum())

# data types
print("\nDtypes and non-null counts:")
aviation_df.info()

# calculate % of missing values
print("\nMissing values (% of rows) per column:")
print(aviation_df.isna().mean().mul(100).round(1).sort_values(ascending=False))

# summary statistics
print("--------------------")
print("Summary Statistics")
print("--------------------")
aviation_df.describe()

Shape: (88889, 31)
Duplicate Event.Ids: 938

Dtypes and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Numbe

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


**Observations**
- The file is not UTF-8, so it is read with `encoding="latin1"`.
- Several columns (e.g. `Latitude`, `Longitude`, `Airport.Code`, `Air.carrier`, `Schedule`) are mostly empty and will be dropped later.
- The four injury columns have NaNs. In NTSB data a blank usually means "none", but we only trust that when at least one of the four counts is recorded (see the injury section).
- `Event.Date` is stored as text and must be converted to a datetime before filtering by year.

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [6]:
clean_df = aviation_df.copy()
row_log = [("Raw data", len(clean_df))]

# Make column names consistent with the lab text (the raw file uses 'Aircraft.damage')
clean_df.columns = clean_df.columns.str.strip()
clean_df = clean_df.rename(columns={"Aircraft.damage": "Aircraft.Damage"})

# 1. Remove duplicate events
clean_df = clean_df.drop_duplicates(subset="Event.Id")
row_log.append(("Drop duplicate Event.Id", len(clean_df)))

# 2. Parse dates and keep events from 1983 onwards (40-year max aircraft lifetime)
clean_df["Event.Date"] = pd.to_datetime(clean_df["Event.Date"], errors="coerce")
clean_df = clean_df.dropna(subset=["Event.Date"])
clean_df["Event.Year"] = clean_df["Event.Date"].dt.year
clean_df = clean_df[clean_df["Event.Year"] >= 1983]
row_log.append(("Events from 1983 onwards", len(clean_df)))

# 3. Keep accidents only (incidents rarely involve damage/injury and are not the client's focus)
clean_df["Investigation.Type"] = clean_df["Investigation.Type"].str.strip().str.title()
clean_df = clean_df[clean_df["Investigation.Type"] == "Accident"]
row_log.append(("Accidents only", len(clean_df)))

# 4. Professional builds only: drop amateur-built; rows with unknown build status are dropped
#    because we cannot confirm they are professional builds
clean_df["Amateur.Built"] = clean_df["Amateur.Built"].str.strip().str.title()
clean_df = clean_df[clean_df["Amateur.Built"] == "No"]
row_log.append(("Professional builds only", len(clean_df)))

# 5. Airplanes only. Aircraft.Category is blank for many older records, so impute it from
#    other records of the same Make + Model whose category is known (most common category).
cat = clean_df["Aircraft.Category"].str.strip().str.title()
clean_df["Aircraft.Category"] = cat.mask(cat.isin(["Unk", "Unknown", ""]))
print("Missing Aircraft.Category before imputation:", clean_df["Aircraft.Category"].isna().sum())

clean_df["_key"] = clean_df["Make"].str.upper().str.strip() + "|" + clean_df["Model"].str.upper().str.strip()
key_to_category = (
    clean_df.dropna(subset=["Aircraft.Category", "_key"])
            .groupby("_key")["Aircraft.Category"]
            .agg(lambda s: s.mode().iloc[0])
)
clean_df["Aircraft.Category"] = clean_df["Aircraft.Category"].fillna(clean_df["_key"].map(key_to_category))
print("Missing Aircraft.Category after imputation: ", clean_df["Aircraft.Category"].isna().sum())

print("\nCategory counts:")
print(clean_df["Aircraft.Category"].value_counts(dropna=False).head(8))

clean_df = clean_df[clean_df["Aircraft.Category"] == "Airplane"].drop(columns="_key").copy()
row_log.append(("Airplanes only", len(clean_df)))

pd.DataFrame(row_log, columns=["Step", "Rows remaining"])

Missing Aircraft.Category before imputation: 48814
Missing Aircraft.Category after imputation:  7343

Category counts:
Aircraft.Category
Airplane             57422
NaN                   7343
Helicopter            6437
Glider                 770
Balloon                295
Weight-Shift           139
Powered Parachute       83
Gyrocraft               31
Name: count, dtype: int64


,Step,Rows remaining
0,Raw data,88889
1,Drop duplicate Event.Id,87951
2,Events from 1983 onwards,84397
3,Accidents only,80753
4,Professional builds only,72557
5,Airplanes only,57422


**Filtering decisions**
1. Duplicate `Event.Id` rows removed.
2. Events before 1983 removed (40-year maximum lifetime assumption).
3. Only `Accident` investigations kept (not `Incident`).
4. Amateur-built aircraft removed. Rows with unknown build status are also removed because they cannot be confirmed as professional builds.
5. Only airplanes kept (no helicopters, gliders, balloons, etc.). Missing categories were imputed from the most common category of the same Make + Model; records that could not be resolved were dropped.

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [7]:
injury_cols = ["Total.Fatal.Injuries", "Total.Serious.Injuries", "Total.Minor.Injuries", "Total.Uninjured"]
clean_df[injury_cols] = clean_df[injury_cols].apply(pd.to_numeric, errors="coerce")

# If all four counts are blank there is no occupant information -> drop.
# Otherwise a blank count means zero people in that category -> fill with 0.
no_info = clean_df[injury_cols].isna().all(axis=1)
print("Rows with no injury information (dropped):", no_info.sum())
clean_df = clean_df[~no_info].copy()
clean_df[injury_cols] = clean_df[injury_cols].fillna(0)

# Estimated number of people on board = fatal + serious + minor + uninjured
clean_df["Total.Passengers"] = clean_df[injury_cols].sum(axis=1)
print("Rows with zero recorded occupants (dropped):", (clean_df["Total.Passengers"] == 0).sum())
clean_df = clean_df[clean_df["Total.Passengers"] > 0].copy()

# Derived measures
clean_df["Total.Fatal.Serious"] = clean_df["Total.Fatal.Injuries"] + clean_df["Total.Serious.Injuries"]
clean_df["Fatal.Serious.Rate"] = clean_df["Total.Fatal.Serious"] / clean_df["Total.Passengers"]
clean_df["Fatal.Rate"] = clean_df["Total.Fatal.Injuries"] / clean_df["Total.Passengers"]

print("\nRows remaining:", len(clean_df))
clean_df[["Total.Passengers", "Total.Fatal.Serious", "Fatal.Serious.Rate", "Fatal.Rate"]].describe()

Rows with no injury information (dropped): 39
Rows with zero recorded occupants (dropped): 279

Rows remaining: 57104


,Total.Passengers,Total.Fatal.Serious,Fatal.Serious.Rate,Fatal.Rate
count,57104.000000,57104.000000,57104.000000,57104.000000
mean,4.204802,0.792011,0.277675,0.179817
std,20.052566,5.089966,0.432330,0.373905
min,1.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000
50%,2.000000,0.000000,0.000000,0.000000
75%,2.000000,1.000000,0.800000,0.000000
max,576.000000,295.000000,1.000000,1.000000


**Injury measures and assumptions**
- Blank injury counts are treated as 0 only when at least one of the four counts was recorded. Rows where all four are blank have no occupant information and are dropped.
- `Total.Passengers` = fatal + serious + minor + uninjured. It is an estimate of everyone on board (crew included), since the data does not separate crew from passengers.
- `Total.Fatal.Serious` = fatal + serious injuries.
- `Fatal.Serious.Rate` = `Total.Fatal.Serious / Total.Passengers`, the share of occupants killed or seriously injured in an accident (0 to 1).
- `Fatal.Rate` = `Total.Fatal.Injuries / Total.Passengers`.
- Rows with zero recorded occupants are dropped because a rate cannot be computed.

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [8]:
print("Raw values:")
print(clean_df["Aircraft.Damage"].value_counts(dropna=False))

# Standardise text and treat unknown placeholders as missing
dmg = clean_df["Aircraft.Damage"].str.strip().str.title()
clean_df["Aircraft.Damage"] = dmg.mask(dmg.isin(["Unknown", "Unk", ""]))

# Damage level is needed for the destruction rate, so rows without it are dropped
print("\nRows with unknown damage (dropped):", clean_df["Aircraft.Damage"].isna().sum())
clean_df = clean_df.dropna(subset=["Aircraft.Damage"]).copy()

# Derived flag: 1 = destroyed, 0 = not destroyed (substantial or minor damage)
clean_df["Destroyed"] = (clean_df["Aircraft.Damage"] == "Destroyed").astype(int)

print("\nCleaned values:")
print(clean_df["Aircraft.Damage"].value_counts())
print("\nShare of aircraft destroyed:", round(clean_df["Destroyed"].mean(), 3))

Raw values:
Aircraft.Damage
Substantial    44348
Destroyed      11607
NaN              750
Minor            335
Unknown           64
Name: count, dtype: int64

Rows with unknown damage (dropped): 814

Cleaned values:
Aircraft.Damage
Substantial    44348
Destroyed      11607
Minor            335
Name: count, dtype: int64

Share of aircraft destroyed: 0.206


**Aircraft damage**
- Text standardised (whitespace and casing); `Unknown` treated as missing and those rows dropped.
- `Destroyed` is a 0/1 flag (1 = `Aircraft.Damage` is "Destroyed"). Its mean over a group is the destruction rate for that group.

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [9]:
clean_df = clean_df.dropna(subset=["Make"]).copy()
raw_unique_makes = clean_df["Make"].nunique()

# Rules applied (in order) to the normalised text; first match wins
MAKE_RULES = [
    (r"^(BOEING STEARMAN|STEARMAN)", "STEARMAN"),     # vintage biplanes, not Boeing airliners
    (r"^BOEING", "BOEING"),
    (r"^CESSNA", "CESSNA"),
    (r"^PIPER", "PIPER"),
    (r"^HAWKER BEECH", "HAWKER BEECHCRAFT"),
    (r"^BEECH", "BEECH"),
    (r"^AIRBUS", "AIRBUS"),
    (r"^MC ?DONNELL", "MCDONNELL DOUGLAS"),
    (r"^DE ?HAVILLAND", "DE HAVILLAND"),
    (r"^(EMBRAER|EMPRESA BRASILEIRA)", "EMBRAER"),
    (r"^(BOMBARDIER|CANADAIR)", "BOMBARDIER"),
    (r"^(LEAR ?JET|GATES LEARJET)", "LEARJET"),
    (r"^GULFSTREAM", "GULFSTREAM"),
    (r"^MOONEY", "MOONEY"),
    (r"^CIRRUS", "CIRRUS"),
    (r"^DIAMOND", "DIAMOND"),
    (r"^MAULE", "MAULE"),
    (r"^GRUMMAN", "GRUMMAN"),
    (r"^AERONCA", "AERONCA"),
    (r"^LUSCOMBE", "LUSCOMBE"),
    (r"^TAYLORCRAFT", "TAYLORCRAFT"),
    (r"^PILATUS", "PILATUS"),
    (r"^SAAB", "SAAB"),
    (r"^FOKKER", "FOKKER"),
    (r"^RAYTHEON", "RAYTHEON"),
]

def clean_make(raw):
    if pd.isna(raw):
        return np.nan
    s = str(raw).upper().strip()
    s = re.sub(r"[-_/]", " ", s)                 # hyphens/slashes -> space
    s = re.sub(r"[^\w\s]", "", s)                # remove other punctuation
    s = re.sub(r"\s+", " ", s).strip()            # collapse whitespace
    s = re.sub(r"^THE\s+", "", s)                 # leading "THE"
    s = re.sub(r"(\s+(INC|CORP|CORPORATION|CO|COMPANY|LTD|LLC|LP|PLC|SA|GMBH))+$", "", s)  # corporate suffixes
    for pattern, canonical in MAKE_RULES:
        if re.match(pattern, s):
            return canonical
    return s if s else np.nan

clean_df["Make"] = clean_df["Make"].map(clean_make)
clean_df = clean_df.dropna(subset=["Make"]).copy()
print(f"Unique makes: {raw_unique_makes} before -> {clean_df['Make'].nunique()} after cleaning")

# Look for near-duplicate spellings that the rules did not catch (inspection only)
make_counts = clean_df["Make"].value_counts()
candidates = make_counts[make_counts >= 10].index.tolist()
pairs = []
for i, a in enumerate(candidates):
    for b in candidates[i + 1:]:
        if difflib.SequenceMatcher(None, a, b).ratio() >= 0.85:
            pairs.append((a, b, make_counts[a], make_counts[b]))
pd.DataFrame(pairs, columns=["Make A", "Make B", "Count A", "Count B"])

Unique makes: 1298 before -> 843 after cleaning


,Make A,Make B,Count A,Count B
0,AMERICAN CHAMPION AIRCRAFT,AMERICAN CHAMPION ACAC,53,18
1,CUB CRAFTERS,CUBCRAFTERS,38,30
2,AMERICAN CHAMPION ACAC,AMERICAN CHAMPION,18,15
3,WACO CLASSIC AIRCRAFT,CLASSIC AIRCRAFT,14,12


**Make cleaning tasks**
1. Drop rows with a missing `Make`.
2. Convert to upper case, replace hyphens/slashes with spaces, remove other punctuation and extra whitespace.
3. Remove a leading "THE" and trailing corporate suffixes (INC, CORP, CO, COMPANY, LTD, ...).
4. Merge spelling variants of the same manufacturer with explicit rules (e.g. `Cessna Aircraft Co` -> `CESSNA`, `Mcdonnell Douglas` -> `MCDONNELL DOUGLAS`, `Canadair` -> `BOMBARDIER`). Boeing-Stearman biplanes are kept separate from Boeing.
5. Check for remaining near-duplicates (table above). Add any you want to merge to `MANUAL_MERGES` below.
6. Keep only makes with at least `MIN_MAKE_ACCIDENTS` accidents so comparisons between makes are not driven by tiny samples.

In [10]:
# Optional: merge extra near-duplicates spotted in the table above, e.g. {"CESNA": "CESSNA"}
MANUAL_MERGES = {}
clean_df["Make"] = clean_df["Make"].replace(MANUAL_MERGES)

# Minimum accidents per make. The lab suggests 50, but lower can work: large-jet makes have few
# accidents in this data. Stricter thresholds can still be applied in the analysis notebook.
MIN_MAKE_ACCIDENTS = 30

make_counts = clean_df["Make"].value_counts()
keep_makes = make_counts[make_counts >= MIN_MAKE_ACCIDENTS].index
print(f"Makes before filter: {len(make_counts)}, kept (>= {MIN_MAKE_ACCIDENTS} accidents): {len(keep_makes)}")

clean_df = clean_df[clean_df["Make"].isin(keep_makes)].copy()
print("Rows remaining:", len(clean_df))
clean_df["Make"].value_counts()

Makes before filter: 843, kept (>= 30 accidents): 63
Rows remaining: 54174


Make
CESSNA                            24209
PIPER                             13311
BEECH                              4292
GRUMMAN                            1329
MOONEY                             1194
                                  ...  
LOCKHEED                             33
COSTRUZIONI AERONAUTICHE TECNA       33
REPUBLIC                             32
EVEKTOR AEROTECHNIK AS               31
CUBCRAFTERS                          30
Name: count, Length: 63, dtype: int64

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [11]:
# 1. Remove NaNs and placeholder labels
clean_df = clean_df.dropna(subset=["Model"]).copy()

def clean_model(raw):
    s = str(raw).upper().strip()
    s = re.sub(r"\s+", " ", s)          # collapse whitespace
    s = re.sub(r"\s*-\s*", "-", s)      # "737 - 200" -> "737-200"
    return s

clean_df["Model"] = clean_df["Model"].map(clean_model)
clean_df = clean_df[~clean_df["Model"].isin(["", "UNKNOWN", "UNK", "NONE", "N/A", "NA"])].copy()

# 2. Model family: the base designation without sub-variant suffixes
#    e.g. 737-824 -> 737, 172N -> 172, PA-28-180 -> PA28, DC-9-82 -> DC9, A320-214 -> A320
FAMILY_RE = re.compile(r"^([A-Z]{1,3})?[\s\-]?(\d{1,4})")
STRIP_PREFIX = {"BOEING": "B", "CESSNA": "C", "BEECH": "BE"}   # e.g. "C172" and "172" are the same Cessna

def model_family(make, model):
    m = FAMILY_RE.match(model)
    if not m:
        return model                      # names such as "KING AIR 200" or "G-IV" are kept as they are
    letters, digits = (m.group(1) or ""), m.group(2)
    if letters == STRIP_PREFIX.get(make):
        letters = ""
    return letters + digits

clean_df["Model.Family"] = [model_family(mk, md) for mk, md in zip(clean_df["Make"], clean_df["Model"])]

# 3. Are model labels unique to each make?
makes_per_family = clean_df.groupby("Model.Family")["Make"].nunique()
shared = makes_per_family[makes_per_family > 1].sort_values(ascending=False)
print(f"{len(shared)} of {len(makes_per_family)} model families appear under more than one make")
print(shared.head(8))

# 4. Unique plane identifier = Make + Model family
clean_df["Aircraft.Type"] = clean_df["Make"] + " " + clean_df["Model.Family"]

print("\nUnique Model values:", clean_df["Model"].nunique())
print("Unique Aircraft.Type values:", clean_df["Aircraft.Type"].nunique())
clean_df["Aircraft.Type"].value_counts().head(15)

83 of 647 model families appear under more than one make
Model.Family
S2     8
500    4
400    4
58     4
7      4
690    4
8      4
35     3
Name: Make, dtype: int64

Unique Model values: 2095
Unique Aircraft.Type values: 771


Aircraft.Type
CESSNA 172      6084
PIPER PA28      4410
CESSNA 150      2916
CESSNA 182      2485
CESSNA 152      2181
PIPER PA18      1563
PIPER PA32      1376
MOONEY M20      1187
CESSNA 180      1001
GRUMMAN G164     864
PIPER PA22       825
CESSNA T210      719
CESSNA 210       701
PIPER PA24       697
CESSNA 177       673
Name: count, dtype: int64

**Model cleaning**
- Rows with a missing or placeholder `Model` were dropped; labels were upper-cased and whitespace/hyphen spacing standardised.
- Model labels are **not** unique to a make (the same number or code appears under several manufacturers), so `Make` + `Model` must be used together.
- `Model.Family` strips sub-variant suffixes so that near-identical aircraft share a sample (e.g. `172`, `172N`, `172S` -> `172`). `Aircraft.Type` = `Make` + `Model.Family` is the unique plane identifier. The exact cleaned `Model` is kept if finer detail is wanted. Names without a numeric designation keep their full cleaned label.

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [12]:
cols = ["Engine.Type", "Weather.Condition", "Number.of.Engines", "Purpose.of.flight", "Broad.phase.of.flight"]

print("BEFORE CLEANING")
for c in cols:
    print(f"\n--- {c} (missing: {clean_df[c].isna().sum()}) ---")
    print(clean_df[c].value_counts(dropna=False).head(12))

BEFORE CLEANING

--- Engine.Type (missing: 2727) ---
Engine.Type
Reciprocating    48299
NaN               2727
Turbo Prop        2043
Turbo Fan          607
Unknown            348
Turbo Jet          127
Turbo Shaft         11
UNK                  1
Name: count, dtype: int64

--- Weather.Condition (missing: 1540) ---
Weather.Condition
VMC    47802
IMC     4247
NaN     1540
UNK      451
Unk      123
Name: count, dtype: int64

--- Number.of.Engines (missing: 1963) ---
Number.of.Engines
1.0    45791
2.0     6206
NaN     1963
0.0       77
4.0       66
3.0       60
Name: count, dtype: int64

--- Purpose.of.flight (missing: 2160) ---
Purpose.of.flight
Personal               32807
Instructional           7630
Aerial Application      3135
Unknown                 2587
Business                2581
NaN                     2160
Positioning              881
Ferry                    474
Other Work Use           473
Aerial Observation       360
Executive/corporate      277
Public Aircraft          244

In [13]:
# Engine.Type: standardise spelling, keep only recognised engine types (others -> NaN)
engine_map = {"Turbofan": "Turbo Fan", "Geared Turbofan": "Turbo Fan", "Turboprop": "Turbo Prop",
              "Turbojet": "Turbo Jet", "Turboshaft": "Turbo Shaft"}
valid_engines = ["Reciprocating", "Turbo Prop", "Turbo Fan", "Turbo Jet", "Turbo Shaft", "Electric"]
eng = clean_df["Engine.Type"].str.strip().str.title().replace(engine_map)
clean_df["Engine.Type"] = eng.where(eng.isin(valid_engines))

# Weather.Condition: VMC / IMC only; "UNK" and other placeholders -> NaN
wx = clean_df["Weather.Condition"].str.strip().str.upper()
clean_df["Weather.Condition"] = wx.where(wx.isin(["VMC", "IMC"]))

# Number.of.Engines: numeric; a powered airplane has 1-4 engines, so 0 or other values -> NaN
n_eng = pd.to_numeric(clean_df["Number.of.Engines"], errors="coerce")
clean_df["Number.of.Engines"] = n_eng.where(n_eng.between(1, 4))

# Purpose.of.flight: standardise text, "Unknown" -> NaN, merge the Public Aircraft variants
pur = clean_df["Purpose.of.flight"].str.strip().str.title()
pur = pur.mask(pur.isin(["Unknown", "Unk", ""]))
clean_df["Purpose.of.flight"] = pur.where(~pur.str.startswith("Public Aircraft", na=False), "Public Aircraft")

# Broad.phase.of.flight: standardise text, "Unknown" -> NaN
ph = clean_df["Broad.phase.of.flight"].str.strip().str.title()
clean_df["Broad.phase.of.flight"] = ph.mask(ph.isin(["Unknown", "Unk", ""]))

# Light imputation for engine info: use the most common value for the same Aircraft.Type
# (a given plane type almost always has the same engine type / engine count)
def group_mode(s):
    m = s.mode()
    return m.iloc[0] if len(m) else np.nan

for c in ["Engine.Type", "Number.of.Engines"]:
    before = clean_df[c].isna().sum()
    mode_by_type = clean_df.groupby("Aircraft.Type")[c].agg(group_mode)
    clean_df[c] = clean_df[c].fillna(clean_df["Aircraft.Type"].map(mode_by_type))
    print(f"{c}: missing {before} -> {clean_df[c].isna().sum()} after imputing from Aircraft.Type")

clean_df["Number.of.Engines"] = clean_df["Number.of.Engines"].astype("Int64")

print("\nAFTER CLEANING")
for c in cols:
    print(f"\n--- {c} (missing: {clean_df[c].isna().sum()}) ---")
    print(clean_df[c].value_counts(dropna=False).head(12))

Engine.Type: missing 3076 -> 59 after imputing from Aircraft.Type
Number.of.Engines: missing 2040 -> 45 after imputing from Aircraft.Type

AFTER CLEANING

--- Engine.Type (missing: 59) ---
Engine.Type
Reciprocating    50685
Turbo Prop        2334
Turbo Fan          943
Turbo Jet          131
NaN                 59
Turbo Shaft         11
Name: count, dtype: int64

--- Weather.Condition (missing: 2114) ---
Weather.Condition
VMC    47802
IMC     4247
NaN     2114
Name: count, dtype: int64

--- Number.of.Engines (missing: 45) ---
Number.of.Engines
1       47073
2        6865
4          93
3          87
<NA>       45
Name: count, dtype: Int64

--- Purpose.of.flight (missing: 4747) ---
Purpose.of.flight
Personal               32807
Instructional           7630
NaN                     4747
Aerial Application      3135
Business                2581
Positioning              881
Ferry                    474
Other Work Use           473
Aerial Observation       360
Public Aircraft          328
Exe

**Cleaning tasks for these columns**
- `Engine.Type`: standardised casing and spelling (e.g. `Turbofan` -> `Turbo Fan`); `Unknown`, `None` and unrecognised values set to NaN; missing values filled from the most common engine type of the same `Aircraft.Type`.
- `Weather.Condition`: only `VMC` (visual) and `IMC` (instrument) are valid; `UNK` and other placeholders set to NaN.
- `Number.of.Engines`: converted to numeric; 0 or implausible values set to NaN (a powered airplane has 1-4 engines); missing values filled from the most common count for the same `Aircraft.Type`.
- `Purpose.of.flight`: casing standardised, `Unknown` -> NaN, `Public Aircraft - Federal/State/Local` merged into `Public Aircraft`.
- `Broad.phase.of.flight`: casing standardised, `Unknown` -> NaN.
- Remaining NaNs in these columns are left as is; they can be excluded group by group during analysis.

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [14]:
missing_pct = clean_df.isna().mean().mul(100).round(1).sort_values(ascending=False)
print(missing_pct)

MISSING_THRESHOLD = 50   # drop columns with more than 50% missing values

# Columns needed for the analysis are never dropped
protected = ["Engine.Type", "Weather.Condition", "Number.of.Engines", "Purpose.of.flight",
             "Broad.phase.of.flight", "Make", "Model", "Model.Family", "Aircraft.Type",
             "Aircraft.Damage", "Destroyed", "Total.Passengers", "Fatal.Serious.Rate", "Fatal.Rate"]

cols_to_drop = [c for c in clean_df.columns
                if clean_df[c].isna().mean() * 100 > MISSING_THRESHOLD and c not in protected]
print("\nDropping:", cols_to_drop)
clean_df = clean_df.drop(columns=cols_to_drop).reset_index(drop=True)

print("Final shape:", clean_df.shape)

Schedule                  90.1
Air.carrier               83.2
FAR.Description           68.1
Longitude                 60.3
Latitude                  60.3
Airport.Code              39.0
Airport.Name              36.2
Broad.phase.of.flight     29.5
Publication.Date          16.4
Purpose.of.flight          8.8
Report.Status              5.4
Weather.Condition          3.9
Registration.Number        0.9
Number.of.Engines          0.1
Country                    0.1
Engine.Type                0.1
Event.Date                 0.0
Investigation.Type         0.0
Accident.Number            0.0
Event.Id                   0.0
Location                   0.0
Aircraft.Damage            0.0
Injury.Severity            0.0
Amateur.Built              0.0
Model                      0.0
Aircraft.Category          0.0
Make                       0.0
Total.Uninjured            0.0
Total.Minor.Injuries       0.0
Total.Serious.Injuries     0.0
Total.Fatal.Injuries       0.0
Event.Year                 0.0
Total.Pa

Columns with more than 50% missing values (typically location coordinates, airport details, `Air.carrier` and `Schedule`) carry too little information to be useful and are dropped. The columns used for the analysis are protected from removal.

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [15]:
clean_df.shape

(54163, 34)

In [16]:
OUTPUT_PATH = "./data/AviationData_cleaned.csv"
clean_df.to_csv(OUTPUT_PATH, index=False)

# Reload to confirm the file was written correctly
check_df = pd.read_csv(OUTPUT_PATH, parse_dates=["Event.Date"], low_memory=False)
print("Saved and reloaded:", check_df.shape)
check_df[["Make", "Model", "Aircraft.Type", "Total.Passengers", "Fatal.Serious.Rate", "Destroyed"]].head()

Saved and reloaded: (54163, 34)


,Make,Model,Aircraft.Type,Total.Passengers,Fatal.Serious.Rate,Destroyed
0,CESSNA,182P,CESSNA 182,4.0,0.0,0
1,CESSNA,182RG,CESSNA 182,2.0,0.0,0
2,CESSNA,182P,CESSNA 182,1.0,0.0,0
3,PIPER,PA-28R-200,PIPER PA28,2.0,0.0,0
4,CESSNA,140,CESSNA 140,2.0,0.0,0
